<a href="https://colab.research.google.com/github/BubuDavid/ml-journey-workshop/blob/main/notebooks/02_intro_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción a Redes Neuronales - ML Journey

## ¿Qué son las Redes Neuronales?

Las redes neuronales son algoritmos de machine learning inspirados en el funcionamiento del cerebro humano. Están compuestas por "neuronas" artificiales interconectadas que pueden aprender patrones complejos en los datos.

### ¿Por qué necesitamos Redes Neuronales?

La regresión lineal funciona bien para problemas **lineales**, pero ¿qué pasa cuando nuestros datos tienen patrones **no lineales**? Aquí es donde brillan las redes neuronales.

### Conceptos Clave:

1. **Neurona/Perceptrón**: Unidad básica que recibe inputs, los procesa y produce un output
2. **Capas (Layers)**:
   - **Input Layer**: Recibe los datos
   - **Hidden Layers**: Procesan la información
   - **Output Layer**: Produce el resultado final
3. **Funciones de Activación**: Introducen no-linealidad (ReLU, Sigmoid, Tanh)
4. **Pesos y Bias**: Parámetros que la red aprende durante el entrenamiento

### Ventajas de las Redes Neuronales:
- Pueden aprender patrones complejos y no lineales
- Son versátiles para diferentes tipos de problemas
- Pueden aproximar cualquier función (teorema de aproximación universal)

## Ejemplo Práctico: Problema No Lineal

Vamos a resolver un problema que la regresión lineal **no puede** resolver: clasificar círculos concéntricos.

In [ ]:
# Importar cosas
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
import seaborn as sns

# Configuramos el estilo de las gráficas
plt.style.use('seaborn-v0_8')
np.random.seed(42)  # Para reproducibilidad

In [ ]:
def plot_dataset(X, y, title="Dataset No Lineal", figsize=(12, 5), model=None, padding=0.1):
    """
    Función para visualizar el dataset y opcionalmente la frontera de decisión de un modelo

    Args:
        X: Features (características) del dataset
        y: Labels (etiquetas) del dataset
        title: Título de la gráfica
        figsize: Tamaño de la figura
        model: Modelo entrenado (opcional) para mostrar la frontera de decisión
        padding: Espaciado alrededor de los datos
    """
    fig, axes = plt.subplots(1, 1, figsize=figsize)

    # Colores para cada clase
    colors = ['#FF6B6B', '#4ECDC4']  # Rojo y turquesa

    # Dibujamos cada clase con su color correspondiente
    for i, color in enumerate(colors):
        mask = y == i  # Máscara para filtrar puntos de la clase i
        axes.scatter(X[mask, 0], X[mask, 1],
                    c=color, alpha=0.7, s=30,
                    label=f'Clase {i}', edgecolors='black', linewidth=0.5)

    # Configuración de la gráfica
    axes.set_xlabel('Característica 1')
    axes.set_ylabel('Característica 2')
    axes.set_title(f'{title}')
    axes.legend()
    axes.grid(True, alpha=0.3)
    axes.set_aspect('equal')  # Aspect ratio 1:1

    # Si se proporciona un modelo, dibujamos la frontera de decisión
    if model:
        h = 0.02  # Resolución de la malla
        x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
        y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding

        # Creamos una malla de puntos
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                            np.arange(y_min, y_max, h))
        mesh_points = np.c_[xx.ravel(), yy.ravel()]

        # Predecimos para cada punto de la malla
        Z = model.predict(mesh_points)
        Z = Z.reshape(xx.shape)

        # Dibujamos las regiones de decisión
        axes.contourf(xx, yy, Z, levels=50, alpha=0.4, cmap='RdYlBu')

        # Dibujamos la frontera de decisión
        decision_boundary = axes.contour(
            xx, yy, Z, levels=[0.5], colors='black',
            linestyles='--', linewidths=2
        )
        axes.set_title(f'{title} - con Frontera de Decisión')

    plt.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
# Generamos un dataset de círculos concéntricos (problema no lineal)
X, y = make_circles(n_samples=800, noise=0.1, factor=0.4)
plot_dataset(X, y, "Círculos Concéntricos - Problema No Lineal");

# PRIMER INTENTO: RED NEURONAL SIN CAPAS OCULTAS (PERCEPTRÓN)

In [ ]:
import tensorflow as tf

In [ ]:
print("=== INTENTO 1: Red sin capas ocultas (equivalente a regresión logística) ===")

# Modelo con solo una neurona (sin capas ocultas)
model = tf.keras.Sequential([
    tf.keras.layers.Input((2,)),  # Input: 2 características
    tf.keras.layers.Dense(1, activation='sigmoid')  # Output: 1 neurona con sigmoid
])
# Compilamos el modelo
model.compile(optimizer='adam',           # Algoritmo de optimización
              loss='binary_crossentropy', # Función de pérdida para clasificación binaria
              metrics=['accuracy'])       # Métrica a monitorear

# Entrenamos el modelo
print("Entrenando modelo simple...")
model.fit(X, y, epochs=100, verbose=0)
plot_dataset(X, y, "Intento 1: Sin Capas Ocultas", model=model);

In [ ]:
# Mostramos la arquitectura del modelo
print("\nArquitectura del Modelo 1:")
tf.keras.utils.plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

# SEGUNDO INTENTO: RED NEURONAL CON UNA CAPA OCULTA

In [ ]:
print("\n=== INTENTO 2: Red con una capa oculta ===")

# Modelo con una capa oculta
model = tf.keras.Sequential([
    tf.keras.layers.Input((2,)),                    # Input: 2 características
    tf.keras.layers.Dense(2, activation='relu'),    # Capa oculta: 2 neuronas con ReLU
    tf.keras.layers.Dense(1, activation='sigmoid')  # Output: 1 neurona con sigmoid
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Entrenando modelo con capa oculta...")
model.fit(X, y, epochs=100, verbose=0)

# Visualizamos el resultado
plot_dataset(X, y, "Intento 2: Con Una Capa Oculta", model=model);

In [ ]:
print("\nArquitectura del Modelo 2:")
tf.keras.utils.plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)

# TERCERA ALTERNATIVA: RED MUY PROFUNDA (MUCHAS CAPAS)

In [ ]:
print("\n=== INTENTO 3: Red muy profunda (¿será mejor?) ===")

# Modelo con muchas capas ocultas
model = tf.keras.Sequential([
    tf.keras.layers.Input((2,)),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),
    tf.keras.layers.Dense(2, activation='relu'),  # 10 capas ocultas!
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Entrenando modelo muy profundo...")
model.fit(X, y, epochs=100, verbose=0)

plot_dataset(X, y, "Intento 3: Red Muy Profunda", model=model);

In [ ]:
print("\nArquitectura del Modelo 3:")
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

# CUARTA TENTATIVA: RED CON MÁS NEURONAS EN LA CAPA OCULTA

In [ ]:
print("\n=== INTENTO 4: Más neuronas en una sola capa oculta ===")

# Modelo con una capa oculta pero más neuronas
model = tf.keras.Sequential([
    tf.keras.layers.Input((2,)),                     # Input: 2 características
    tf.keras.layers.Dense(64, activation='relu'),    # Capa oculta: 64 neuronas
    tf.keras.layers.Dense(1, activation='sigmoid')   # Output: 1 neurona
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Entrenando modelo con más neuronas...")
model.fit(X, y, epochs=100, verbose=0)

plot_dataset(X, y, "Intento 4: Más Neuronas por Capa", model=model);

In [ ]:
print("\nArquitectura del Modelo 4:")
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
print("\n=== CONCLUSIONES ===")
print("💡 Observaciones importantes:")
print("1. Un perceptrón simple no puede resolver algunos problemas no lineales")
print("2. Agregar capas ocultas permite aprender patrones no lineales")
print("3. Más capas no siempre es mejor (puede causar overfitting)")
print("4. A veces es mejor tener más neuronas que más capas")
print("5. La arquitectura óptima depende del problema específico")

# Enlace recomendado para experimentar
print("\n🔗 Recomendación:")
print("Puedes experimentar con diferentes arquitecturas en:")
print("https://playground.tensorflow.org/")

# EJERCICIO PRÁCTICO: FASHION MNIST
Fashion MNIST es un dataset de imágenes de ropa y accesorios. Es perfecto para practicar clasificación de imágenes con redes neuronales.

### Características del Dataset:
- **60,000** imágenes de entrenamiento
- **10,000** imágenes de prueba  
- **10 clases** diferentes de ropa
- Imágenes en **escala de grises** de **28x28 píxeles**

### Las 10 clases son:
0. T-shirt/top
1. Trouser (Pantalón)
2. Pullover (Suéter)
3. Dress (Vestido)
4. Coat (Abrigo)
5. Sandal (Sandalia)
6. Shirt (Camisa)
7. Sneaker (Tenis)
8. Bag (Bolsa)
9. Ankle boot (Botín)

In [ ]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Cargamos el dataset Fashion MNIST
fashion_mnist = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = fashion_mnist.load_data()

In [ ]:
# Configuramos numpy para mostrar arrays más legibles
np.set_printoptions(linewidth=200)
# Diccionario con los nombres de las clases
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("=== EXPLORACIÓN DEL DATASET ===")
print(f"Imágenes de entrenamiento: {training_images.shape}")
print(f"Etiquetas de entrenamiento: {training_labels.shape}")
print(f"Imágenes de prueba: {test_images.shape}")
print(f"Etiquetas de prueba: {test_labels.shape}")

In [ ]:
# Visualizamos una imagen de ejemplo
img_index = 100  # Índice de la imagen a visualizar

plt.figure(figsize=(8, 6))
plt.imshow(training_images[img_index], cmap='gray')
plt.axis('off')
plt.title(f'Imagen {img_index}: {class_names[training_labels[img_index]]}')
plt.show()

print(f'Etiqueta numérica: {training_labels[img_index]}')
print(f'Clase: {class_names[training_labels[img_index]]}')
print(f'Forma de la imagen: {training_images[img_index].shape}')
print(f'Valores de píxeles (muestra):')
print(training_images[img_index])

# PREPROCESAMIENTO DE DATOS

In [ ]:
print("\n=== PREPROCESAMIENTO ===")
print(f"Rango original de píxeles: {training_images.min()} - {training_images.max()}")

# Normalizamos las imágenes dividiendo por 255 (valores entre 0 y 1)
training_images = training_images / 255.0
test_images = test_images / 255.0

print(f"Rango después de normalización: {training_images.min()} - {training_images.max()}")
print("✅ Normalización completada")

# CREACIÓN Y ENTRENAMIENTO DEL MODELO

In [ ]:
# Creamos un modelo de red neuronal multicapa (MLP)
mlp_model = tf.keras.models.Sequential([
    tf.keras.layers.Input((28, 28)),

    # Capa 1: Aplanamos las imágenes 28x28 a vectores de 784 elementos
    tf.keras.layers.Flatten(),

    # Capa 2: Capa oculta con 256 neuronas y activación ReLU
    tf.keras.layers.Dense(256, activation='relu'),

    # Capa 3: Capa de salida con 10 neuronas (una por clase) y activación softmax
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compilamos el modelo
mlp_model.compile(
    optimizer=tf.optimizers.SGD(),              # Optimizador: Stochastic Gradient Descent
    loss='sparse_categorical_crossentropy',     # Función de pérdida para clasificación multiclase
    metrics=['accuracy']                        # Métrica a monitorear
)

tf.keras.utils.plot_model(mlp_model, show_shapes=True, show_layer_names=True);

In [ ]:
print("\n=== ENTRENAMIENTO ===")
print("Entrenando modelo... (esto puede tomar unos minutos)")

# Entrenamos el modelo
history = mlp_model.fit(
    training_images,
    training_labels,
    epochs=3,        # Número de épocas (pasadas completas por todo el dataset)
    verbose=1        # Muestra el progreso del entrenamiento
)

# EVALUACIÓN DEL MODELO

In [ ]:
print("\n=== EVALUACIÓN ===")

# Evaluamos el modelo en el conjunto de prueba
test_loss, test_accuracy = mlp_model.evaluate(test_images, test_labels, verbose=0)
print(f"Precisión en el conjunto de prueba: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
import random
print("\n=== HACIENDO PREDICCIONES ===")

# Seleccionamos una imagen aleatoria del conjunto de prueba
test_index = random.randint(0, len(test_images) - 1)

# Visualizamos la imagen
plt.figure(figsize=(8, 6))
plt.imshow(test_images[test_index], cmap='gray')
plt.axis('off')
plt.title(f'Imagen de Prueba #{test_index}')
plt.show()

# Preparamos la imagen para la predicción
# Necesitamos agregar una dimensión extra para el batch
input_image = np.expand_dims(test_images[test_index], axis=0)

# Hacemos la predicción
prediction = mlp_model.predict(input_image, verbose=0)

# La predicción es un array con las probabilidades para cada clase
predicted_class = np.argmax(prediction)
confidence = prediction[0][predicted_class]

print(f"Etiqueta real: {test_labels[test_index]} ({class_names[test_labels[test_index]]})")
print(f"Predicción: {predicted_class} ({class_names[predicted_class]})")
print(f"Confianza: {confidence:.4f} ({confidence*100:.2f}%)")

In [ ]:
# Mostramos las probabilidades para todas las clases
print(f"\nProbabilidades para todas las clases:")
for i, prob in enumerate(prediction[0]):
    print(f"{class_names[i]}: {prob:.4f} ({prob*100:.1f}%)")

In [ ]:
# Verificamos si la predicción es correcta
if predicted_class == test_labels[test_index]:
    print("✅ ¡Predicción CORRECTA!")
else:
    print("❌ Predicción incorrecta")

print("\n=== RESUMEN DEL EJERCICIO ===")
print("🎯 Lo que hemos logrado:")
print("1. Cargamos y exploramos el dataset Fashion MNIST")
print("2. Preprocesamos las imágenes (normalización)")
print("3. Creamos una red neuronal multicapa")
print("4. Entrenamos el modelo con 60,000 imágenes")
print("5. Evaluamos el rendimiento en 10,000 imágenes de prueba")
print("6. Hicimos predicciones en imágenes nuevas")
print(f"7. Obtuvimos una precisión de {test_accuracy*100:.1f}% ¡No está mal!")